# Clasificacion de Sign Language MNIST con MLP

Notebook principal del proyecto. La idea es ejecutar un flujo completo: carga de datos, EDA, preprocesamiento, entrenamiento, evaluacion y analisis de errores.

## 1. Contexto del problema

El problema consiste en clasificar imagenes de letras del lenguaje de senas americano. Cada imagen representa una letra y el modelo debe predecir la clase correcta. Como primera version se usa un Perceptron Multicapa, aunque se espera discutir sus limitaciones frente a imagenes.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.model_selection import train_test_split

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJECT_ROOT / "src"))

from sign_mlp.data import as_images, label_to_letter, load_sign_mnist, make_label_mapping, remap_labels
from sign_mlp.evaluate import plot_confusion, plot_history, plot_prediction_examples, report_metrics
from sign_mlp.model import build_mlp

sns.set_theme(style="whitegrid")
PROJECT_ROOT

## 2. Carga de datos

Antes de ejecutar esta seccion, correr desde la raiz del repositorio:

```powershell
python scripts\prepare_data.py --zip "C:\Users\gonza\Downloads\Sign Language MNIST.zip"
```

In [ ]:
X_train_full, y_train_full, X_test, y_test = load_sign_mnist(PROJECT_ROOT / "data" / "raw")

print("Train:", X_train_full.shape, y_train_full.shape)
print("Test:", X_test.shape, y_test.shape)
print("Pixel min/max:", X_train_full.min(), X_train_full.max())

## 3. Analisis exploratorio de datos

In [ ]:
labels_df = pd.DataFrame({"label": y_train_full})
labels_df["letter"] = labels_df["label"].apply(label_to_letter)

plt.figure(figsize=(12, 4))
sns.countplot(data=labels_df, x="letter", order=sorted(labels_df["letter"].unique()))
plt.title("Distribucion de clases en entrenamiento")
plt.xlabel("Letra")
plt.ylabel("Cantidad de imagenes")
plt.tight_layout()
plt.show()

In [ ]:
unique_labels = sorted(np.unique(y_train_full))
fig, axes = plt.subplots(4, 6, figsize=(12, 8))
axes = axes.ravel()

for ax, label in zip(axes, unique_labels[:24]):
    idx = np.where(y_train_full == label)[0][0]
    ax.imshow(as_images(X_train_full[[idx]])[0], cmap="gray")
    ax.set_title(label_to_letter(label))
    ax.axis("off")

for ax in axes[len(unique_labels[:24]):]:
    ax.axis("off")

plt.suptitle("Ejemplos por clase")
plt.tight_layout()
plt.show()

## 4. Preprocesamiento

Las imagenes ya vienen como vectores de 784 pixeles. Cada pixel se normalizo dividiendo por 255, por lo que los valores quedan entre 0 y 1. Ademas, se crea una particion de validacion desde el set de entrenamiento.

In [ ]:
RANDOM_STATE = 42

label_to_index, index_to_label = make_label_mapping(y_train_full, y_test)
y_train_full_idx = remap_labels(y_train_full, label_to_index)
y_test_idx = remap_labels(y_test, label_to_index)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_full,
    y_train_full_idx,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y_train_full_idx,
)

num_classes = len(index_to_label)
print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)
print("Classes:", num_classes)

## 5. Modelo MLP

Arquitectura base: entrada de 784 pixeles, capas densas con activacion ReLU, dropout para reducir sobreajuste y salida softmax para clasificacion multiclase.

In [ ]:
model = build_mlp(
    input_dim=X_train.shape[1],
    num_classes=num_classes,
    hidden_layers=(256, 128),
    dropout=0.25,
    learning_rate=0.001,
)

model.summary()

## 6. Entrenamiento

In [ ]:
history = model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=20,
    batch_size=128,
    verbose=1,
)

In [ ]:
fig = plot_history(history)
fig.savefig(PROJECT_ROOT / "images" / "training_curves.png", dpi=150)
plt.show()

## 7. Evaluacion de desempeno

In [ ]:
test_loss, test_accuracy = model.evaluate(X_test, y_test_idx, verbose=0)
print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_accuracy:.4f}")

y_proba = model.predict(X_test)
y_pred = np.argmax(y_proba, axis=1)

metrics_text = report_metrics(y_test_idx, y_pred, index_to_label)
print(metrics_text)

(PROJECT_ROOT / "reports" / "classification_report.txt").write_text(metrics_text, encoding="utf-8")

In [ ]:
fig = plot_confusion(y_test_idx, y_pred, index_to_label)
fig.savefig(PROJECT_ROOT / "images" / "confusion_matrix.png", dpi=150)
plt.show()

## 8. Analisis de errores

In [ ]:
fig = plot_prediction_examples(X_test, y_test_idx, y_pred, index_to_label, correct=True, max_items=9)
fig.savefig(PROJECT_ROOT / "images" / "correct_examples.png", dpi=150)
plt.show()

fig = plot_prediction_examples(X_test, y_test_idx, y_pred, index_to_label, correct=False, max_items=9)
fig.savefig(PROJECT_ROOT / "images" / "incorrect_examples.png", dpi=150)
plt.show()

## 9. Guardado del modelo

In [ ]:
model.save(PROJECT_ROOT / "models" / "sign_language_mlp.keras")

## 10. Conclusiones para completar

- Desempeno alcanzado: completar con accuracy y F1-score final.
- Principales aciertos: completar despues de revisar matriz de confusion.
- Principales errores: completar con clases mas confundidas.
- Limitaciones del MLP: pierde estructura espacial de la imagen al aplanarla; no aprende filtros locales como una CNN.
- Mejoras futuras: CNN, data augmentation, tuning de hiperparametros, regularizacion y validacion mas robusta.